# 5b — PopPUNK concordance and antimicrobial resistance enrichment

Reproduces the population-structure validation and the AMR analysis added during revision:

| Output | Manuscript location |
|---|---|
| Phylon vs PopPUNK concordance (ARI, AMI, homogeneity) | Results, "Phylons recapitulate established GAS population structure" |
| Per-phylon phylogroup breakdown | Table S11b |
| AMR carriage by phylon (chi-square, Fisher/BH) | Results, "Distinct GAS lineages encode alternative virulence architectures" |
| Per-genome supplementary table | Table S10 |
| Accessory-gene intersection structure | Table S12, Fig. S8a |
| Zero-phylon frequency distribution | Fig. S8b |

**Inputs** (`INPUT_DIR`, default `../../../data`) — matrices from the Zenodo deposit
(DOI 10.5281/zenodo.20403112) plus the PopPUNK and Bakta outputs generated below:

    L_bin_panta_cdhit_Spyo.csv          binarized L (1,167 genes x 11 phylons)
    A_bin_panta_cdhit_Spyo.csv          binarized A (11 phylons x 399 genomes)
    df_acc_panta_cdhit.pickle           accessory P matrix (1,167 x 399)
    car_metadata_Sypo.csv               per-genome metadata (399 x 137)
    GAS_assigned_clusters.csv           PopPUNK reference assignment
    GAS_refined_clusters.csv            PopPUNK de novo fit
    Spyo_bakta_annot_expert_long.tsv    Bakta expert-system hits
    Spyo_bakta_annot_fct_long.tsv       FCT/pilin-locus gene content

In [ ]:
from pathlib import Path
import collections, re
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, fisher_exact, chi2_contingency
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import (adjusted_rand_score, adjusted_mutual_info_score,
                             normalized_mutual_info_score,
                             homogeneity_completeness_v_measure)

INPUT_DIR  = Path("../../../data")     # adjust if your matrices live elsewhere
OUTPUT_DIR = Path("../../../results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PHYLONS = ['emm-patternE-other','ST28-emm1','ST52-emm28','ST39-emm4','emm3/6/5',
           'ST36','ST63-emm77','SpeK-prophage','emm-patternD-other',
           'SpeC-prophage','ST101-emm89']
MOBILONS = {'SpeK-prophage','SpeC-prophage'}
TAXONOMIC = [p for p in PHYLONS if p not in MOBILONS]          # 9
NAMED     = ['ST28-emm1','ST52-emm28','ST39-emm4','emm3/6/5','ST36',
             'ST63-emm77','ST101-emm89']                        # 7

## 1. Generating the PopPUNK assignments

Run once, outside this notebook. PopPUNK v2.7.8; the reference database is
`Streptococcus_pyogenes_v1_refs` (2,083 genomes; Davies et al., *Nat Genet* 2019),
downloaded from <https://www.bacpop.org/poppunk-databases/>.

```bash
# rfiles.txt: one line per genome, "<genome_id>\t<absolute path to .fna>"
python - <<'EOF'
import csv, os
A = 'A_bin_panta_cdhit_Spyo.csv'
with open(A) as f: ids = next(csv.reader(f))[1:]
with open('rfiles.txt','w') as out:
    for g in ids:
        out.write(f"{g}\t{os.path.abspath(f'bakta/{g}/{g}.fna')}\n")
EOF

cd poppunk-outputs          # run from inside; PopPUNK's plotting breaks on
                            # --output paths containing a directory separator
cp rfiles.txt qfiles.txt

# (a) assignment against the public reference database  -> used in the paper
poppunk_assign --db /path/to/gas_references --query qfiles.txt \
               --output GAS_assigned --threads 64

# (b) de novo fit from these 399 genomes               -> reported for completeness
poppunk --create-db  --r-files rfiles.txt --output GAS_db      --threads 64
poppunk --fit-model dbscan --ref-db GAS_db --output GAS_dbscan --threads 64
poppunk --fit-model refine --ref-db GAS_db --model-dir GAS_dbscan \
        --output GAS_refined --threads 64
```

Four pairs of reference phylogroups merge during assignment (7/97, 35/96, 87/199,
39/218) because query genomes fall within the refined boundary separating them;
merged assignments carry composite identifiers.

In [ ]:
L = pd.read_csv(INPUT_DIR / "L_bin_panta_cdhit_Spyo.csv", index_col=0)
A = pd.read_csv(INPUT_DIR / "A_bin_panta_cdhit_Spyo.csv", index_col=0)
if L.shape[0] == 11 and L.shape[1] != 11:
    L = L.T
L.columns = PHYLONS
A.index   = PHYLONS
genomes   = list(A.columns)

meta = pd.read_csv(INPUT_DIR / "car_metadata_Sypo.csv", low_memory=False).set_index("genome_id")
acc  = pd.read_pickle(INPUT_DIR / "df_acc_panta_cdhit.pickle")

def load_poppunk(path):
    d = pd.read_csv(path)
    d.columns = ["Taxon", "Cluster"]
    return dict(zip(d.Taxon.astype(str), d.Cluster.astype(str)))

# PopPUNK writes taxon names with '.' replaced by '_'
key      = {g: g.replace(".", "_") for g in genomes}
assigned = load_poppunk(INPUT_DIR / "GAS_assigned_clusters.csv")
refined  = load_poppunk(INPUT_DIR / "GAS_refined_clusters.csv")
for name, d in [("reference-assigned", assigned), ("de novo", refined)]:
    missing = [g for g in genomes if key[g] not in d]
    assert not missing, f"{name}: {len(missing)} genomes unmatched, e.g. {missing[:3]}"

pp_ref = np.array([assigned[key[g]] for g in genomes])
pp_dn  = np.array([refined[key[g]]  for g in genomes])
print(f"{len(genomes)} genomes | {len(set(pp_ref))} reference phylogroups | "
      f"{len(set(pp_dn))} de novo clusters")

## 2. Phylon labels

Taxonomic phylons are mutually exclusive by construction — no genome carries two —
so a single label per genome is well defined. Mobilons are overlays and are excluded
from the concordance comparison.

In [ ]:
Atax = A.loc[TAXONOMIC]
labels = []
for g in genomes:
    hits = list(Atax.index[Atax[g] > 0])
    assert len(hits) <= 1, f"{g} carries >1 taxonomic phylon: {hits}"
    labels.append(hits[0] if hits else "Unassigned")
phylon = np.array(labels)

mlst = meta.reindex(genomes)["mlst"].astype(str).values
raw  = [str(x) for x in meta.reindex(genomes)["Predicted emm-type"].tolist()]
emm  = np.array([x.replace("EMM", "").split(".")[0] for x in raw])

print(collections.Counter(phylon).most_common())
print(f"\nsingly assigned: {(phylon != 'Unassigned').sum()} / {len(genomes)}")

## 3. Concordance — Table S11a

Reported at two scopes. The nine-phylon numbers are lower because the two composite
phylons aggregate many unrelated lineages by design; showing both forestalls the
objection that only the favourable scope was reported.

**Homogeneity is the metric that matters here.** A value of 1.000 means every cluster of
the comparison partition falls entirely inside a single phylon — the nesting claim.
Completeness is correspondingly low because a phylon contains many phylogroups.

In [ ]:
def metrics(truth, pred, scope, comparison):
    h, c, v = homogeneity_completeness_v_measure(truth, pred)
    return dict(Scope=scope, Comparison=comparison, n=len(truth),
                n_classes=len(set(pred)),
                ARI=round(adjusted_rand_score(truth, pred), 3),
                AMI=round(adjusted_mutual_info_score(truth, pred), 3),
                NMI=round(normalized_mutual_info_score(truth, pred), 3),
                Homogeneity=round(h, 3), Completeness=round(c, 3),
                V_measure=round(v, 3))

COMPARISONS = [("PopPUNK genomic phylogroup (reference-assigned)", pp_ref),
               ("PopPUNK cluster (de novo)", pp_dn),
               ("MLST sequence type", mlst),
               ("emm base type", emm)]

rows = []
for scope, mask in [("7 lineage-specific phylons", np.isin(phylon, NAMED)),
                    ("9 taxonomic phylons",        phylon != "Unassigned")]:
    for name, vec in COMPARISONS:
        rows.append(metrics(phylon[mask], vec[mask], scope, name))

S11a = pd.DataFrame(rows)
S11a.to_csv(OUTPUT_DIR / "TableS11a_global_concordance_PopPUNK.csv", index=False)
S11a

In [ ]:
# Containment: are PopPUNK clusters strictly nested within phylons?
m  = phylon != "Unassigned"
df = pd.DataFrame({"pp": pp_ref[m], "phy": phylon[m]})
grp   = df.groupby("pp")["phy"].agg(["nunique", "count"])
multi = grp[grp["count"] >= 2]
print(f"{len(grp)} phylogroups among assigned genomes; "
      f"{(multi['nunique'] == 1).sum()}/{len(multi)} multi-genome phylogroups "
      f"fall entirely within one phylon")
for cl in grp.index[grp["nunique"] > 1]:
    sub = df[df.pp == cl]
    print(f"  phylogroup {cl}: n={len(sub)} spans {dict(collections.Counter(sub.phy))}")

## 4. Per-phylon phylogroup breakdown — Table S11b

In [ ]:
rows = []
for p in TAXONOMIC:
    s = df[df.phy == p]
    if s.empty:
        continue
    idx = np.where(phylon == p)[0]
    vr, vd = pd.Series(pp_ref[idx]).value_counts(), pd.Series(pp_dn[idx]).value_counts()
    rows.append(dict(
        Phylon=p, n_strains=len(idx),
        n_PopPUNK_phylogroups_ref=int(pd.Series(pp_ref[idx]).nunique()),
        dominant_phylogroup=vr.index[0], dominant_phylogroup_n=int(vr.iloc[0]),
        dominant_phylogroup_pct=round(100 * vr.iloc[0] / len(idx), 1),
        phylogroups_contained_in_this_phylon_only=int(
            sum(grp.loc[c, "nunique"] == 1 for c in pd.unique(pp_ref[idx]))),
        n_PopPUNK_clusters_denovo=int(pd.Series(pp_dn[idx]).nunique()),
        denovo_largest_pct=round(100 * vd.iloc[0] / len(idx), 1),
        n_MLST_STs=int(pd.Series(mlst[idx]).nunique()),
        n_emm_base_types=int(pd.Series(emm[idx]).nunique())))

S11b = pd.DataFrame(rows).sort_values("n_strains", ascending=False)
S11b.to_csv(OUTPUT_DIR / "TableS11b_per_phylon_PopPUNK.csv", index=False)
S11b

## 5. Antimicrobial resistance

Gene content comes from the AMRFinderPlus component of the Bakta annotation. Extract it
first with:

```bash
python scripts/extract_bakta_annotations.py /path/to/bakta/ Spyo_bakta_annot
```

AMRFinderPlus entries in the Bakta JSON carry `"type": "amrfinder"` and **no** `source`
key — filtering on `source` silently returns nothing.

In [ ]:
expert = pd.read_csv(INPUT_DIR / "Spyo_bakta_annot_expert_long.tsv", sep="\t", dtype=str)
fct    = pd.read_csv(INPUT_DIR / "Spyo_bakta_annot_fct_long.tsv",    sep="\t", dtype=str)

G   = set(genomes)                       # restrict to the final 399 (extraction ran on 402)
amr = expert[(expert.category == "amrfinder") & expert.genome_id.isin(G)]
print(f"{len(amr)} AMRFinderPlus hits across {amr.genome_id.nunique()} genomes")
print(amr.gene.value_counts().to_string())

In [ ]:
CLASS = {**{g: "tetracycline" for g in ["tet(M)","tet(O)","tet(T)","tet(L)","tet(K)"]},
         **{g: "macrolide-lincosamide-streptogramin B"
            for g in ["erm(A)","erm(B)","erm(C)","erm(T)"]},
         **{g: "macrolide efflux" for g in ["mef(A)","mef(J)","msr(D)","msr(I)"]},
         **{g: "aminoglycoside"
            for g in ["aph(3')-IIIa","ant(6)-Ia","ant(9)-Ib","aac(6')-Ie/aph(2'')-Ia"]},
         **{g: "trimethoprim" for g in ["dfrG","dfrF"]},
         "sat4": "streptothricin", "catA": "phenicol",
         "thfT": "folate transport (non-resistance)"}

def join(s):
    return ";".join(sorted({str(x) for x in s if isinstance(x, str) and x and x != "nan"}))

amr_genes = amr.groupby("genome_id")["gene"].apply(join)
amr_n     = amr.groupby("genome_id")["gene"].nunique()
amr_cls   = (amr.assign(c=amr.gene.map(CLASS).fillna("other"))
                .groupby("genome_id")["c"].apply(join))
amr_det   = (amr.assign(d=amr.gene + "(" +
                 (amr.identity.astype(float) * 100).round(1).astype(str) + "%)")
                .groupby("genome_id")["d"].apply(join))

carriers = amr.genome_id.nunique()
print(f"{carriers}/{len(genomes)} genomes ({100*carriers/len(genomes):.1f}%) carry "
      f">=1 determinant; {amr.gene.nunique()} distinct genes")

### 5.1 Enrichment

The chi-square test across the nine taxonomic phylons is the primary, pre-specified test.
The per-determinant Fisher tests are descriptive follow-ups and are BH-corrected across
the full gene x phylon grid, matching the treatment of the Sda1 prophage core (Table S7).

In [ ]:
tbl = pd.DataFrame({"genome_id": genomes, "phylon": phylon})
tbl["n_amr"] = amr_n.reindex(tbl.genome_id).fillna(0).astype(int).values
tbl["amr_genes"] = amr_genes.reindex(tbl.genome_id).fillna("").values
tax = tbl[tbl.phylon != "Unassigned"]

ct = pd.crosstab(tax.phylon, tax.n_amr > 0)
chi2, p_chi, dof, _ = chi2_contingency(ct)
print(f"AMR carriage x phylon: chi2 = {chi2:.1f}, df = {dof}, P = {p_chi:.3g}\n")

summary = (tax.assign(has=tax.n_amr > 0)
              .groupby("phylon")
              .agg(n=("has", "size"), n_with_AMR=("has", "sum")))
summary["pct"] = (100 * summary.n_with_AMR / summary.n).round(1)
print(summary.sort_values("pct", ascending=False).to_string())

In [ ]:
genes_tested = sorted({g for s in tbl.amr_genes if s for g in s.split(";")})
rows = []
for gene in genes_tested:
    has = tbl.amr_genes.str.contains(gene, regex=False).values
    for ph in TAXONOMIC:
        inp = tbl.phylon.values == ph
        a, b = int((has & inp).sum()), int((has & ~inp).sum())
        c, d = int((~has & inp).sum()), int((~has & ~inp).sum())
        if a == 0:
            continue
        orr, pv = fisher_exact([[a, b], [c, d]], alternative="greater")
        rows.append(dict(gene=gene, phylon=ph, carriers_in_phylon=a, n_phylon=a + c,
                         carriers_elsewhere=b, odds_ratio=round(orr, 1), p_value=pv))

enr = pd.DataFrame(rows)
enr["q_value"] = multipletests(enr.p_value, method="fdr_bh")[1]
enr = enr.sort_values("p_value")
enr.to_csv(OUTPUT_DIR / "AMR_enrichment_by_phylon.csv", index=False)
enr[enr.q_value < 0.05].head(12)

## 6. Table S10 — per-genome characteristics

In [ ]:
def mcol(name):
    return ([str(x) for x in meta.reindex(genomes)[name].tolist()]
            if name in meta.columns else [""] * len(genomes))

S10 = pd.DataFrame({
    "genome_id": genomes,
    "bioproject_accession": mcol("bioproject_accession"),
    "phylon_label": phylon,
    "n_taxonomic_phylons_assigned": Atax[genomes].sum(axis=0).values,
    "SpeC_mobilon": A.loc["SpeC-prophage", genomes].values,
    "SpeK_mobilon": A.loc["SpeK-prophage", genomes].values,
    "poppunk_ref_cluster": pp_ref,
    "poppunk_denovo_cluster": pp_dn,
    "mlst": mlst,
    "emm_base_type": emm,
    "emm_subtype": raw,
    "emm_cluster": mcol("emm cluster"),
    "country": mcol("country_merged"),
    "collection_date": mcol("collection_date"),
    "isolation_source": mcol("isolation_source"),
    "source_cat": mcol("source_cat"),
    "checkm_completeness": mcol("checkm_completeness"),
    "checkm_contamination": mcol("checkm_contamination"),
})
S10["n_amr_genes"] = amr_n.reindex(genomes).fillna(0).astype(int).values
S10["amr_genes"]   = amr_genes.reindex(genomes).fillna("").values
S10["amr_classes"] = amr_cls.reindex(genomes).fillna("").values
S10["amr_detail_pct_identity"] = amr_det.reindex(genomes).fillna("").values
S10["pilin_locus_genes"] = (fct[fct.genome_id.isin(G)].groupby("genome_id")["gene"]
                            .apply(join).reindex(genomes).fillna("").values)
S10["vfdb_genes"] = (expert[(expert.category == "VFDB") & expert.genome_id.isin(G)]
                     .groupby("genome_id")["gene"].apply(join)
                     .reindex(genomes).fillna("").values)
S10["n_IS_elements"] = (expert[(expert.category == "IS") & expert.genome_id.isin(G)]
                        .groupby("genome_id").size().reindex(genomes).fillna(0)
                        .astype(int).values)

S10.to_csv(OUTPUT_DIR / "TableS10_per_genome_399.csv", index=False)
print(S10.shape)
S10.head(3)

## 7. Table S12 and Figure S8 — intersection structure

In [ ]:
Lacc  = L.reindex(acc.index)
combo = Lacc.apply(lambda r: tuple(sorted(Lacc.columns[r > 0])), axis=1)
cnt   = {k: v for k, v in collections.Counter(combo).items() if k}
deg   = collections.Counter(len(k) for k in cnt)

print(f"{len(cnt)} distinct non-empty combinations; "
      f"{sum(v for k, v in deg.items() if k >= 4)} are four-way or higher")
print(f"zero-phylon genes: {collections.Counter(combo)[()]}")

S12 = (pd.Series({"|".join(k): v for k, v in cnt.items()})
         .sort_values(ascending=False).rename("Genes in intersection"))
S12.index.name = "Phylon combination"
S12.to_csv(OUTPUT_DIR / "TableS12_phylon_intersections.csv")

sda1 = tuple(sorted(("ST28-emm1", "ST36", "ST63-emm77")))
print(f"\nSda1 prophage core as an exact three-way intersection: {cnt[sda1]} genes")

In [ ]:
freq = acc.sum(axis=1)
zero = Lacc.sum(axis=1) == 0
u, p = mannwhitneyu(freq[zero], freq[~zero], alternative="two-sided")
dec  = pd.qcut(freq, 10, labels=False, duplicates="drop") + 1

print(f"zero-phylon n={int(zero.sum())}, median freq {freq[zero].median():.0f} "
      f"vs {freq[~zero].median():.0f} assigned (Mann-Whitney P = {p:.3g})")
print(f"within 2x of the accessory-rare boundary: "
      f"{100*(freq[zero] <= 66).mean():.0f}% vs {100*(freq[~zero] <= 66).mean():.0f}%")
print("deciles occupied by zero-phylon genes:",
      sorted(dec[zero].unique()))

Render the figure itself with the standalone script, which keeps the plotting code out of
the notebook:

```bash
python scripts/make_figure_s8.py --indir data --outdir results
```